# 01. Quickstart & Custom Themes

Welcome to `xkcd-matchem`! The library is built around a simple workflow: **start with clean default presets in one line, then customize anything you like** once you know what story your figure should tell.

In this first notebook, you will:
- Inspect the **15 bundled sample files** and **6 built-in theme presets**.
- Make your first plot in a single function call.
- Compare light and dark chalkboard presets side by side inside a standard `plt.subplots()` layout.
- Register your own reusable theme preset, series palette, and element color map.

:::{note}
`xkcd-matchem` uses subtle xkcd-style line wiggle and bundled **Patrick Hand** typography (SIL OFL 1.1) for playful exploration, posters, slides, and science communication. For archival journal figures—especially sharp spectroscopic peak fitting—stick to standard straight-line Matplotlib plots.
:::

## Basic Usage

Let's see what ships right inside the package. Notice the six built-in presets: `classic` (open white canvas), `notebook` (subtle gridlines), `poster` (bolder strokes and higher DPI), `minimal` (lighter wiggle), `dark_chalk` (chalkboard slide theme), and `blueprint` (engineering cyanotype blue).

In [ ]:
import matplotlib.pyplot as plt
import xkcd_matchem as xm

print("Version:", xm.__version__)
print("Bundled samples:", len(xm.list_samples()), "->", xm.list_samples()[:8])
print("Built-in presets:", xm.list_presets())
print("Built-in element schemes:", xm.list_element_palettes())

Any bundled sample filename (such as `"nacl.cif"` or `"aspirin.xyz"`) can be passed directly to a plotter—no path boilerplate needed. Here is a simulated $\text{Cu K}\alpha_1$ powder XRD pattern using the default `classic` preset:

In [ ]:
ax = xm.plot_xrd("nacl.cif", label_peaks=4, title="Default 'classic' Preset: NaCl XRD")

Often you already have a multi-panel `fig, axes = plt.subplots(...)` layout from Matplotlib and want to style individual panels consistently. Calling `xm.style_existing_axes(ax, preset=...)` updates the background, spines, ticks, and font family together without leaking changes globally:

In [ ]:
x_vals = [0.0, 1.0, 2.0, 3.0, 4.0]
y_vals = [0.2, 1.3, 0.9, 1.8, 1.1]

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.6))
    for ax, name in zip(axes, ("classic", "notebook", "dark_chalk")):
        xm.style_existing_axes(ax, preset=name)
        preset_obj = xm.style.get_preset(name)
        col = "#F0E442" if name == "dark_chalk" else "#0072B2"
        ax.plot(x_vals, y_vals, marker="o", color=col, lw=2.2)
        ax.set_title(f"preset={name!r}", color=preset_obj.ink_color)
        ax.set_xlabel("Reaction Coordinate")
        ax.set_ylabel("Energy (eV)")
        xm.style.apply_xkcd_spines(ax, preset=name)
    fig.tight_layout()

:::{tip}
Notice how `xm.xkcd_style(...)` works as a context manager: once Python leaves the `with` block, your global Matplotlib `rcParams` return cleanly to normal.
:::

## Advanced Usage

Preparing a poster or slide deck with your lab's color identity? You can extend any built-in preset or palette in three clean lines:
1. `xm.register_preset(name, base=..., ...)` creates a reusable `ThemePreset` inheriting defaults from any existing preset.
2. `xm.register_palette(name, [...])` registers a discrete color cycle for curves, bars, and hulls.
3. `xm.register_element_palette(name, {...}, base=...)` overrides specific chemical elements while keeping sensible fallbacks for the rest of the periodic table.

In [ ]:
# 1. Derive a warm sketchbook theme from 'classic'
xm.register_preset(
    "warm_sketchbook",
    base="classic",
    figure_facecolor=xm.style.WARM_PAPER,
    axes_facecolor=xm.style.WARM_PAPER,
    ink_color="#1F2430",
    sketch_scale=1.15,
    grid=True,
    grid_alpha=0.25,
)

# 2. Register custom series colors and an element scheme
xm.register_palette("mineral", ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"])
xm.register_element_palette(
    "terracotta",
    {"Ti": "#E07A5F", "O": "#3D405B", "Si": "#F2CC8F"},
    base="pastel",
)

with xm.xkcd_style("warm_sketchbook", palette="mineral"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.3))
    xm.style_existing_axes(ax1, preset="warm_sketchbook")
    xm.style_existing_axes(ax2, preset="warm_sketchbook")
    xm.plot_phase_diagram_2d(
        [0.0, 0.33, 0.5, 0.67, 1.0],
        [0.0, -0.28, -0.41, -0.14, 0.0],
        labels=["A", "A2B", "AB", "AB2", "B"],
        ax=ax1,
        palette="mineral",
        preset="warm_sketchbook",
        title="Custom Series Palette & Paper Preset",
    )
    xm.plot_crystal(
        "tio2_slab.poscar",
        palette="terracotta",
        preset="warm_sketchbook",
        ax=ax2,
        title="Custom Element Scheme ('terracotta')",
    )
    fig.tight_layout()

:::{tip}
**Try Next:** Swap `sketch_scale=1.15` to `0.4` for a subtler notebook sketch or `1.6` for a livelier comic vibe, then head to **Tutorial 02** to render molecules in 3D, 2D, and the offline browser studio.
:::